In [1]:
!pip install timm faiss-cpu scikit-learn opencv-python huggingface_hub diffusers transformers accelerate opensimplex --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 10.3 MB/s eta 0:00:00


In [2]:
#Cài đặt hoặc cập nhật gdown để tải file từ Google Drive
!pip install --upgrade gdown -q

import gdown
import os

# ID file từ link bạn cung cấp
file_id = '1VMQ1m0g-lulxLdcfqKAwgWc9ClB6QA-_'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'all_sia_data.zip'

# Tải file về Kaggle
gdown.download(url, output, quiet=False)

# Giải nén vào thư mục làm việc
if os.path.exists(output):
    !mkdir -p /kaggle/working/sia_data
    !unzip -q {output} -d /kaggle/working/sia_data
    !rm {output}
    print(" Đã tải và giải nén thành công bộ dữ liệu SIA!")

Downloading...
From (original): https://drive.google.com/uc?id=1VMQ1m0g-lulxLdcfqKAwgWc9ClB6QA-_
From (redirected): https://drive.google.com/uc?id=1VMQ1m0g-lulxLdcfqKAwgWc9ClB6QA-_&confirm=t&uuid=62148cd2-dad1-4953-9af3-bdba1f7fbe96
To: /kaggle/working/all_sia_data.zip
100%|██████████| 2.09G/2.09G [00:27<00:00, 76.2MB/s]


 Đã tải và giải nén thành công bộ dữ liệu SIA!


In [4]:
import os
import random
import time
import numpy as np
import cv2
import json
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
import timm
import faiss
from sklearn.metrics import roc_auc_score, precision_recall_curve
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

# Đăng nhập Hugging Face an toàn qua Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Đăng nhập Hugging Face thành công!")
except Exception as e:
    print("Chưa tìm thấy HF_TOKEN trong Kaggle Secrets. Quá trình lưu mô hình lên HF có thể bị bỏ qua.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

CONFIG = {
    'data_path': '/kaggle/input/datasets/ipythonx/mvtec-ad', 
    'sia_data_path': '/kaggle/working/sia_data', # Đường dẫn thư mục vừa giải nén
    'image_size': 224,
    'batch_size': 64,          
    'coreset_ratio': 0.1,    
    'fine_tune_epochs': 100,    
    'learning_rate': 1e-4,
    'hf_repo_id': "Manh2005/upgrade-version" 
}

try:
    api = HfApi()
    api.create_repo(repo_id=CONFIG['hf_repo_id'], repo_type="model", exist_ok=True)
except:
    pass

MVTEC_CATEGORIES = [
    'carpet', 'grid', 'leather', 'tile', 'wood',
    'bottle', 'cable', 'capsule', 'hazelnut', 'metal_nut', 
    'pill', 'screw', 'toothbrush', 'transistor', 'zipper'
]

Đăng nhập Hugging Face thành công!
Đang sử dụng thiết bị: cpu


In [5]:
import os
import random
import cv2
import numpy as np
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset
from PIL import Image
from opensimplex import OpenSimplex

class MVTecDataset(Dataset):
    def __init__(self, root_dir, sia_dir, category, is_train=True, use_sia=False):
        self.samples = [] 
        self.sia_paths = []
        self.is_train = is_train
        self.use_sia = use_sia
        
        # 1. Nạp ảnh Normal (Good)
        good_dir = os.path.join(root_dir, category, 'train' if is_train else 'test', 'good')
        if os.path.exists(good_dir):
            for f in os.listdir(good_dir):
                if f.endswith(('.png', '.jpg')):
                    self.samples.append({'img': os.path.join(good_dir, f), 'label': 0, 'mask': None})

        # 2. Nạp ảnh Defect thực tế (CHỈ CHO TẬP TEST) -> Sửa lỗi nan AUROC
        if not is_train:
            test_dir = os.path.join(root_dir, category, 'test')
            for d_type in os.listdir(test_dir):
                d_path = os.path.join(test_dir, d_type)
                if not os.path.isdir(d_path) or d_type == 'good': continue
                for f in os.listdir(d_path):
                    if f.endswith(('.png', '.jpg')):
                        m_name = f.rsplit('.', 1)[0] + '_mask.png'
                        m_path = os.path.join(root_dir, category, 'ground_truth', d_type, m_name)
                        self.samples.append({'img': os.path.join(d_path, f), 'label': 1, 'mask': m_path if os.path.exists(m_path) else None})

        # 3. Nạp phôi lỗi SIA cho tập Train
        if is_train and use_sia:
            sia_cat_dir = os.path.join(sia_dir, category)
            if os.path.exists(sia_cat_dir):
                self.sia_paths = [os.path.join(sia_cat_dir, f) for f in os.listdir(sia_cat_dir) if f.endswith(('.png', '.jpg'))]
                # Nhân đôi danh sách để tạo ảnh trộn SIA (label=1)
                orig_samples = list(self.samples)
                for s in orig_samples:
                    self.samples.append({'img': s['img'], 'label': 1, 'mask': None})

        self.transform = T.Compose([
            T.Resize((256, 256)), T.CenterCrop((224, 224)),
            T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        self.mask_transform = T.Compose([T.Resize((224, 224), interpolation=T.InterpolationMode.NEAREST), T.ToTensor()])
        self.gen = OpenSimplex(seed=42)

    def __len__(self):
        return len(self.samples)

    def _generate_simplex_mask(self):
        mask = np.zeros((224, 224), dtype=np.float32)
        scale = random.uniform(50, 100)
        for y in range(224):
            for x in range(224):
                mask[y, x] = self.gen.noise2(x/scale, y/scale)
        mask = ((mask - mask.min()) / (mask.max() - mask.min()) * 255).astype(np.uint8)
        mask = cv2.threshold(mask, 120, 255, cv2.THRESH_BINARY)[1]
        
        focus = np.zeros_like(mask)
        sx, sy = random.randint(50, 120), random.randint(50, 120)
        cv2.rectangle(focus, (sx, sy), (sx+random.randint(40,80), sy+random.randint(40,80)), 255, -1)
        return np.expand_dims(cv2.GaussianBlur(cv2.bitwise_and(mask, focus), (15, 15), 0).astype(np.float32)/255.0, -1)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img_I = Image.open(s['img']).convert('RGB').resize((224, 224))
        
        if self.is_train and s['label'] == 1 and self.sia_paths:
            I_np = np.array(img_I).astype(np.float32)
            P_np = np.array(Image.open(random.choice(self.sia_paths)).convert('RGB').resize((224, 224))).astype(np.float32)
            M = self._generate_simplex_mask()
            delta = random.uniform(0.5, 1.0)
            
            # Equation 3 RealNet
            A_np = ((1.0 - M) * I_np) + ((1.0 - delta) * (M * I_np)) + (delta * (M * P_np))
            final_img = Image.fromarray(A_np.astype(np.uint8))
        else:
            final_img = img_I

        mask = self.mask_transform(Image.open(s['mask']).convert('L')) if s['mask'] else torch.zeros((1, 224, 224))
        return self.transform(final_img), s['label'], mask

In [6]:
class ViTCoreExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        # Trọng số pre-trained ImageNet, nhẹ và hiệu quả
        self.backbone = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=2)
        self.avg_pool = nn.AvgPool2d(kernel_size=3, stride=1, padding=1)
        self._register_hooks()

    def _register_hooks(self):
        def hook_fn(module, input, output):
            # Gắn trực tiếp kết quả vào module hiện tại đang chạy trên GPU đó
            module.extracted_feature = output
            
        self.backbone.layers[2].blocks[2].register_forward_hook(hook_fn)
        self.backbone.layers[2].blocks[3].register_forward_hook(hook_fn)

    def forward_features(self, x):
        _ = self.backbone.forward_features(x)
        
        # Lấy đặc trưng trực tiếp từ các block đã chạy
        feat3 = self.backbone.layers[2].blocks[2].extracted_feature
        feat4 = self.backbone.layers[2].blocks[3].extracted_feature
        
        block3 = feat3.permute(0, 3, 1, 2)
        block4 = feat4.permute(0, 3, 1, 2)
        
        block4_pooled = self.avg_pool(block4)
        return torch.cat([block3, block4_pooled], dim=1)
    
    def forward_classifier(self, x):
        return self.backbone(x)

In [7]:
def k_center_greedy(features, ratio):
    num_samples = int(features.shape[0] * ratio)
    if num_samples == 0: return features
    coreset_idx = [np.random.randint(0, features.shape[0])]
    min_distances = np.linalg.norm(features - features[coreset_idx[0]], axis=1)
    for _ in range(1, num_samples):
        farthest_idx = np.argmax(min_distances)
        coreset_idx.append(farthest_idx)
        new_distances = np.linalg.norm(features - features[farthest_idx], axis=1)
        min_distances = np.minimum(min_distances, new_distances)
    return features[coreset_idx]

def calculate_anomaly_scores(test_features, memory_bank_index, k=9):
    B, C, H, W_dim = test_features.shape 
    test_patches = test_features.view(B, C, H * W_dim).permute(0, 2, 1).reshape(-1, C).cpu().numpy()
    
    distances, _ = memory_bank_index.search(test_patches, k)
    
    # Chặn số âm siêu nhỏ trước khi lấy căn
    distances = np.maximum(distances, 0)
    distances = np.sqrt(distances)
    
    distances_tensor = torch.tensor(distances)
    softmax_weights = F.softmax(distances_tensor, dim=1)
    
    base_scores = distances_tensor[:, 0]
    W = 1.0 - softmax_weights[:, 0]
    
    anomaly_scores_flat = base_scores * W
    anomaly_scores = anomaly_scores_flat.view(B, H * W_dim)
    
    image_scores = anomaly_scores.max(dim=1)[0].numpy()
    patch_scores = anomaly_scores.view(B, H, W_dim).numpy() 
    
    return image_scores, patch_scores

In [8]:
def find_best_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx], f1_scores[best_idx]

In [ ]:
import os
import time
import torch
import torch.nn as nn
import json
import faiss
import cv2
import numpy as np
import gc
from sklearn.metrics import roc_auc_score
from scipy.ndimage import gaussian_filter
from torch.utils.data import DataLoader
import torch.optim as optim

# --- CẤU HÌNH TỐI ƯU ---
CONFIG = {
    'batch_size': 32,
    'learning_rate': 1e-4,
    'fine_tune_epochs': 5,
    'coreset_ratio': 0.1,  # Giảm tỷ lệ coreset nếu vẫn lỗi RAM
    'subsample_ratio': 0.2, # Chỉ lấy 20% patches ngẫu nhiên trước khi chạy Coreset
    'data_path': '/kaggle/input/datasets/ipythonx/mvtec-ad',
    'sia_data_path': '/kaggle/working/sia_data',
    'hf_repo_id': 'your-repo-id'
}

class ViTMultiGPUWrapper(nn.Module):
    def __init__(self, core_model, mode='features'):
        super().__init__()
        self.core_model = core_model
        self.mode = mode

    def forward(self, x):
        if self.mode == 'classifier':
            return self.core_model.forward_classifier(x)
        elif self.mode == 'features':
            return self.core_model.forward_features(x)
        return x

def fine_tune_cutpaste_parallel(model, train_loader, epochs, device):
    model.train()
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    # Sử dụng DataParallel nếu có nhiều GPU
    if torch.cuda.device_count() > 1:
        parallel_model = nn.DataParallel(ViTMultiGPUWrapper(model, mode='classifier'))
    else:
        parallel_model = ViTMultiGPUWrapper(model, mode='classifier')
    
    parallel_model.to(device)
    
    best_loss = float('inf')
    patience = 2
    patience_counter = 0
    
    for epoch in range(epochs):
        total_loss = 0
        for images, labels, _ in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = parallel_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(train_loader)
        print(f"      Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0 
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
                
    # Trả model về trạng thái gốc và giải phóng parallel_model
    model_state = parallel_model.module.core_model.state_dict() if isinstance(parallel_model, nn.DataParallel) else parallel_model.core_model.state_dict()
    model.load_state_dict(model_state)
    del parallel_model
    return model

def build_memory_bank_parallel(model, dataloader, device):
    model.eval()
    temp_bank = []
    
    if torch.cuda.device_count() > 1:
        parallel_model = nn.DataParallel(ViTMultiGPUWrapper(model, mode='features'))
    else:
        parallel_model = ViTMultiGPUWrapper(model, mode='features')
    
    parallel_model.to(device)
        
    with torch.no_grad():
        for images, _, _ in dataloader:
            features = parallel_model(images.to(device))
            # Features shape: [B, C, H, W]
            B, C, H, W = features.shape
            patches = features.view(B, C, H*W).permute(0, 2, 1).reshape(-1, C).cpu().numpy().astype(np.float32)
            
            # Subsampling ngẫu nhiên ngay lập tức để tiết kiệm RAM
            if CONFIG['subsample_ratio'] < 1.0:
                idx = np.random.choice(len(patches), int(len(patches) * CONFIG['subsample_ratio']), replace=False)
                patches = patches[idx]
                
            temp_bank.append(patches)
            
    full_bank = np.concatenate(temp_bank, axis=0)
    del temp_bank
    
    # Coreset sampling (Hàm k_center_greedy cần được định nghĩa bên ngoài)
    coreset = k_center_greedy(full_bank, CONFIG['coreset_ratio'])
    del full_bank
    
    index = faiss.IndexFlatL2(coreset.shape[1])
    index.add(coreset)
    
    del parallel_model
    return index, coreset

# --- VÒNG LẶP CHÍNH ĐÃ TỐI ƯU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
evaluation_results = {}
num_workers = min(4, os.cpu_count()) # Giới hạn worker để tránh ngốn RAM hệ thống

for category in MVTEC_CATEGORIES:
    print(f"\n" + "="*85)
    print(f"XỬ LÝ: {category.upper()} | Workers: {num_workers} | GPUs: {torch.cuda.device_count()}")
    print("="*85)
    
    # 1. Load Data
    ai_dataset = MVTecDataset(CONFIG['data_path'], CONFIG['sia_data_path'], category, is_train=True, use_sia=True)
    normal_dataset = MVTecDataset(CONFIG['data_path'], CONFIG['sia_data_path'], category, is_train=True, use_sia=False)
    test_dataset = MVTecDataset(CONFIG['data_path'], CONFIG['sia_data_path'], category, is_train=False, use_sia=False)
    
    cutpaste_loader = DataLoader(ai_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=num_workers, pin_memory=True)
    normal_loader = DataLoader(normal_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=num_workers, pin_memory=True)
    
    # 2. Model & Fine-tune
    model = ViTCoreExtractor().to(device)
    model = fine_tune_cutpaste_parallel(model, cutpaste_loader, CONFIG['fine_tune_epochs'], device)
    
    # 3. Memory Bank
    faiss_index, _ = build_memory_bank_parallel(model, normal_loader, device)
    
    # 4. Inference
    all_img_scores, all_img_labels = [], []
    # Dùng list để chứa các mảng phẳng, sau đó mới nối một lần duy nhất
    all_pixel_scores_list, all_pixel_labels_list = [], [] 
    
    model.eval()
    start_time = time.time()
    
    with torch.no_grad():
        for images, labels, masks in test_loader:
            # Lấy features (cần bọc trong Wrapper nếu muốn dùng DataParallel ở đây)
            features = model.forward_features(images.to(device))
            img_scores, patch_scores = calculate_anomaly_scores(features, faiss_index)
            
            all_img_scores.extend(img_scores)
            all_img_labels.extend(labels.numpy())
            
            for i in range(len(patch_scores)):
                score_map = cv2.resize(patch_scores[i], (224, 224), interpolation=cv2.INTER_LINEAR)
                score_map = gaussian_filter(score_map, sigma=4)
                
                # Lưu dưới dạng float32 để tiết kiệm bộ nhớ
                all_pixel_scores_list.append(score_map.flatten().astype(np.float32))
                all_pixel_labels_list.append(masks[i].numpy().flatten().astype(np.uint8))
            
    # Chuyển đổi sang numpy một lần duy nhất
    pixel_scores_final = np.concatenate(all_pixel_scores_list)
    pixel_labels_final = np.concatenate(all_pixel_labels_list)
    del all_pixel_scores_list, all_pixel_labels_list # Xóa ngay lập tức

    # Tính toán Metrics
    fps = len(test_dataset) / (time.time() - start_time)
    image_auroc = roc_auc_score(all_img_labels, all_img_scores)
    pixel_auroc = roc_auc_score(pixel_labels_final, pixel_scores_final)
    
    print(f"Image AUROC: {image_auroc:.4f} | Pixel AUROC: {pixel_auroc:.4f}")
    
    # Lưu kết quả
    evaluation_results[category] = {
        'Img_AUROC': float(image_auroc), 'Pix_AUROC': float(pixel_auroc), 'FPS': float(fps),
        'Model_MB': 0, 'Index_MB': 0, 'VRAM_MB': torch.cuda.max_memory_allocated(device) / (1024*1024)
    }

    # --- DỌN DẸP CUỐI CATEGORY (QUAN TRỌNG) ---
    del pixel_scores_final, pixel_labels_final, faiss_index, model
    torch.cuda.empty_cache()
    gc.collect()


XỬ LÝ: CARPET | Workers: 4 | GPUs: 0


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

      Epoch [1/5] - Loss: 0.0852
      Epoch [2/5] - Loss: 0.0000
      Epoch [3/5] - Loss: 0.0000
      Epoch [4/5] - Loss: 0.0000
      Epoch [5/5] - Loss: 0.0000
Image AUROC: 1.0000 | Pixel AUROC: 0.9848

XỬ LÝ: GRID | Workers: 4 | GPUs: 0
      Epoch [1/5] - Loss: 0.1398
      Epoch [2/5] - Loss: 0.0000
      Epoch [3/5] - Loss: 0.0000
      Epoch [4/5] - Loss: 0.0000
      Epoch [5/5] - Loss: 0.0000
Image AUROC: 0.9323 | Pixel AUROC: 0.9250

XỬ LÝ: LEATHER | Workers: 4 | GPUs: 0
      Epoch [1/5] - Loss: 0.0544
      Epoch [2/5] - Loss: 0.0000
      Epoch [3/5] - Loss: 0.0000
      Epoch [4/5] - Loss: 0.0000
      Epoch [5/5] - Loss: 0.0000
Image AUROC: 0.9969 | Pixel AUROC: 0.9916

XỬ LÝ: TILE | Workers: 4 | GPUs: 0
      Epoch [1/5] - Loss: 0.0811
      Epoch [2/5] - Loss: 0.0000
      Epoch [3/5] - Loss: 0.0000
      Epoch [4/5] - Loss: 0.0000
      Epoch [5/5] - Loss: 0.0000
Image AUROC: 0.9953 | Pixel AUROC: 0.9700

XỬ LÝ: WOOD | Workers: 4 | GPUs: 0
      Epoch [1/5] - Loss: